In [1]:
import pandas as pd
train_df = pd.read_parquet(r"final_data\chunk-based-split\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"final_data\chunk-based-split\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"final_data\chunk-based-split\test_df_prepared.parquet", engine="pyarrow")

In [2]:
cols_to_drop = ["flow_id", "timestamp", "src_ip", "src_port", "dst_ip", "dst_port"]
train_df = train_df.drop(columns=cols_to_drop)
valid_df = valid_df.drop(columns=cols_to_drop)
test_df = test_df.drop(columns=cols_to_drop)

In [3]:
X_train, y_train = train_df.drop(columns=["label"]), train_df["label"]
X_valid, y_valid = valid_df.drop(columns=["label"]), valid_df["label"]
X_test, y_test = test_df.drop(columns=["label"]), test_df["label"]

KNN

In [4]:
from sklearn.metrics import f1_score, classification_report
import torch
import numpy as np
import time
import pandas as pd

X_train, y_train = train_df.drop(columns=["label"]), train_df["label"]
X_valid, y_valid = valid_df.drop(columns=["label"]), valid_df["label"]
X_test, y_test = test_df.drop(columns=["label"]), test_df["label"]

class TorchKNNClassifier:
    def __init__(self, n_neighbors=5, val_batch_size=256, train_batch_size=50000, device='cuda'):
        self.k = n_neighbors
        self.val_batch_size = val_batch_size
        self.train_batch_size = train_batch_size
        self.device = device
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        # Chuyển dữ liệu lên GPU một lần để tính toán
        if isinstance(X, pd.DataFrame): X = X.values
        if isinstance(y, pd.Series): y = y.values
        self.X_train = torch.tensor(X, dtype=torch.float32, device=self.device)
        self.y_train = torch.tensor(y, dtype=torch.long, device=self.device)

    def predict(self, X):
        if isinstance(X, pd.DataFrame): X = X.values
        X_val = torch.tensor(X, dtype=torch.float32, device=self.device)
        preds = []
        
        # Chia batch dữ liệu CẦN dự đoán 
        for i in range(0, len(X_val), self.val_batch_size):
            X_batch = X_val[i:i+self.val_batch_size]
            
            # Khởi tạo buffer chứa min distances. Mặc định là vô cực (inf)
            current_top_dists = torch.full((len(X_batch), self.k), float('inf'), device=self.device)
            current_top_indices = torch.zeros((len(X_batch), self.k), dtype=torch.long, device=self.device)
            
            # Phân tách X_train thành các khối nhỏ để lặp qua (Tránh tạo ma trận khổng lồ gây OOM)
            for j in range(0, len(self.X_train), self.train_batch_size):
                X_tr_batch = self.X_train[j:j+self.train_batch_size]
                
                # Tính khoảng cách Euclidean
                dists = torch.cdist(X_batch, X_tr_batch)
                
                # Mức độ cắt k
                k_chunk = min(self.k, dists.shape[1])
                chunk_top_dists, chunk_top_idx = torch.topk(dists, k_chunk, dim=1, largest=False)
                
                # Sửa index thành chỉ số toàn cục trong self.X_train
                chunk_top_idx += j
                
                # Gộp với k nhỏ nhất của các khối trước, sau đó lại top-k một lần nữa
                merged_dists = torch.cat([current_top_dists, chunk_top_dists], dim=1)
                merged_idx = torch.cat([current_top_indices, chunk_top_idx], dim=1)
                
                current_top_dists, best_k_pos = torch.topk(merged_dists, self.k, dim=1, largest=False)
                current_top_indices = torch.gather(merged_idx, 1, best_k_pos)

                # Dọn rác VRAM
                del dists, merged_dists, merged_idx
            
            # Lấy nhãn của k láng giềng gần nhất chung cuộc
            topk_labels = self.y_train[current_top_indices]
            
            # Dự đoán theo nhãn chiếm đa số (mode)
            mode_labels, _ = torch.mode(topk_labels, dim=1)
            preds.append(mode_labels.cpu().numpy())
            
        return np.concatenate(preds)

# Thử các giá trị k khác nhau
k_values = [3, 5, 7]

best_k = None
best_f1 = -1
best_model = None

print("Đang huấn luyện phân lớp KNN (Multi-batch Memory Efficient qua PyTorch) và tìm 'k' tốt nhất...")
start_time = time.time()

for k in k_values:
    # Sử dụng TorchKNNClassifier kết hợp cắt nhỏ ma trận, GPU sẽ an toàn
    knn = TorchKNNClassifier(n_neighbors=k, val_batch_size=256, train_batch_size=50000, device='cuda')
    knn.fit(X_train, y_train)
    
    y_valid_pred = knn.predict(X_valid)
    
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    print(f"k = {k:2d} | Validation Macro F1: {current_f1:.4f}")
    
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_k = k
        best_model = knn

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> K tốt nhất tìm được là k = {best_k} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test
print("\n--- Đánh giá Model Tốt nhất (k={}) trên tập TEST ---".format(best_k))
y_test_pred = best_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))

Đang huấn luyện phân lớp KNN (Multi-batch Memory Efficient qua PyTorch) và tìm 'k' tốt nhất...
k =  3 | Validation Macro F1: 0.7926
k =  5 | Validation Macro F1: 0.7955
k =  7 | Validation Macro F1: 0.7979

=> Quá trình huấn luyện và tuning hoàn tất trong 1078.69 giây.
=> K tốt nhất tìm được là k = 7 với Validation Macro F1: 0.7979

--- Đánh giá Model Tốt nhất (k=7) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.5039    0.9230    0.6519     19851
           1     0.9877    1.0000    0.9938    484073
           2     0.6644    0.8239    0.7356      2521
           3     0.9753    0.8100    0.8850     36260
           4     0.5634    0.6577    0.6069     18981
           5     0.9902    0.9898    0.9900      2460
           6     0.6915    0.7875    0.7364     11851
           7     0.9999    0.9965    0.9982    107440
           8     0.9207    0.3033    0.4563     16760
           9     0.9952    0.8940    0.9419     41522
          10     0

SVM

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report
import time
import numpy as np
import pandas as pd

print("Đang huấn luyện phân lớp Multi-class Linear SVM (chạy trên 100% GPU qua PyTorch) và tìm tham số 'alpha' tốt nhất...")
start_time = time.time()

# Thử các giá trị alpha (L2 regularization)
alpha_values = [1e-3, 1e-4, 1e-5, 1e-6]

best_alpha = None
best_f1 = -1
best_svm_model = None

# Khối 1: Chuyển dữ liệu lên Numpy 
X_trn = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
y_trn = y_train.values if isinstance(y_train, pd.Series) else y_train
X_val = X_valid.values if isinstance(X_valid, pd.DataFrame) else X_valid
X_tst = X_test.values if isinstance(X_test, pd.DataFrame) else X_test

# Khối 2: Đẩy sang Tensor CUDA
device = 'cuda'
X_train_t = torch.tensor(X_trn, dtype=torch.float32, device=device)
# Đối với Multi-class SVM, giữ nguyên nhãn số nguyên (0, 1, 2... 10)
y_train_t = torch.tensor(y_trn, dtype=torch.long, device=device)
X_valid_t = torch.tensor(X_val, dtype=torch.float32, device=device)
X_test_t = torch.tensor(X_tst, dtype=torch.float32, device=device)

input_dim = X_train_t.shape[1]
num_classes = len(torch.unique(y_train_t))

# Kiến trúc Linear SVM (Multi-class) bằng Pytorch
class TorchLinearSVM(nn.Module):
    def __init__(self, in_features, out_classes):
        super().__init__()
        self.linear = nn.Linear(in_features, out_classes) # Sửa output thành 11 classes
        
    def forward(self, x):
        return self.linear(x)

# PyTorch hỗ trợ sẵn MultiMarginLoss dùng để tối ưu Multi-class SVM
criterion = nn.MultiMarginLoss()

for alpha in alpha_values:
    model = TorchLinearSVM(input_dim, num_classes).to(device)
    
    # Khai báo Optimizer - sử dụng weight_decay thay thế L2 penalty
    optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=alpha)
    
    model.train()
    epochs = 100
    batch_size = 8192
    num_samples = X_train_t.shape[0]
    
    for epoch in range(epochs):
        indices = torch.randperm(num_samples, device=device)
        for i in range(0, num_samples, batch_size):
            idx = indices[i:i+batch_size]
            batch_X = X_train_t[idx]
            batch_y = y_train_t[idx]
            
            optimizer.zero_grad()
            outputs = model(batch_X)
            
            # Tính toán trên GPU với Multi-Class Hinge Loss
            loss = criterion(outputs, batch_y)
            
            loss.backward()
            optimizer.step()
    
    # Validation step
    model.eval()
    with torch.no_grad():
        valid_outputs = model(X_valid_t)
        # Sử dụng argmax để lấy chỉ số có logit cao nhất (tương ứng với nhãn)
        y_valid_pred = torch.argmax(valid_outputs, dim=1).cpu().numpy()
        
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    print(f"alpha = {alpha:8.6f} | Validation Macro F1: {current_f1:.4f}")
    
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_alpha = alpha
        best_svm_model = model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: alpha = {best_alpha} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên test
print("\n--- Đánh giá Model Multi-class SVM Tốt nhất (alpha={}) trên tập TEST ---".format(best_alpha))
best_svm_model.eval()
with torch.no_grad():
    test_outputs = best_svm_model(X_test_t)
    y_test_pred_final = torch.argmax(test_outputs, dim=1).cpu().numpy()

print(classification_report(y_test, y_test_pred_final, digits=4))

Đang huấn luyện phân lớp Multi-class Linear SVM (chạy trên 100% GPU qua PyTorch) và tìm tham số 'alpha' tốt nhất...
alpha = 0.001000 | Validation Macro F1: 0.7372
alpha = 0.000100 | Validation Macro F1: 0.7674
alpha = 0.000010 | Validation Macro F1: 0.7485
alpha = 0.000001 | Validation Macro F1: 0.7537

=> Quá trình huấn luyện và tuning hoàn tất trong 126.13 giây.
=> Cấu hình tốt nhất: alpha = 0.0001 với Validation Macro F1: 0.7674

--- Đánh giá Model Multi-class SVM Tốt nhất (alpha=0.0001) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.5029    0.1767    0.2615     19851
           1     0.9995    1.0000    0.9997    484073
           2     0.8226    0.8112    0.8169      2521
           3     0.9771    0.9417    0.9591     36260
           4     0.5031    0.5602    0.5301     18981
           5     0.9936    0.8801    0.9334      2460
           6     0.6954    0.7884    0.7390     11851
           7     0.9998    0.9962    0.9980    107440

Random Forest

In [6]:
from xgboost import XGBRFClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp Random Forest (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...")
start_time = time.time()

# Thử trên các max_depth khác nhau
max_depth_values = [10, 15]

best_depth = None
best_f1 = -1
best_rf_model = None

for depth in max_depth_values:
    # Sử dụng XGBRFClassifier với cấu hình device='cuda' để chạy trên GPU
    rf_model = XGBRFClassifier(
        n_estimators=100, 
        max_depth=depth,
        n_jobs=-1, 
        random_state=42,
        tree_method='hist',
        device='cuda' # Chạy bằng GPU
    )

    # Huấn luyện mô hình
    rf_model.fit(X_train, y_train)
    
    # Dự đoán trên tập validation và tính điểm Macro F1
    y_valid_pred = rf_model.predict(X_valid)
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    
    print(f"max_depth = {depth:2d} | Validation Macro F1: {current_f1:.4f}")
    
    # Chọn model tốt nhất dựa trên Macro F1 của validation
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_depth = depth
        best_rf_model = rf_model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: max_depth = {best_depth} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá mô hình Random Forest Tốt nhất (max_depth={}) trên tập TEST ---".format(best_depth))
y_test_pred = best_rf_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))

Đang huấn luyện phân lớp Random Forest (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:751: UserWarning: [22:44:22] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


max_depth = 10 | Validation Macro F1: 0.8893
max_depth = 15 | Validation Macro F1: 0.9047

=> Quá trình huấn luyện và tuning hoàn tất trong 191.44 giây.
=> Cấu hình tốt nhất: max_depth = 15 với Validation Macro F1: 0.9047

--- Đánh giá mô hình Random Forest Tốt nhất (max_depth=15) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.8052    0.8604    0.8319     19851
           1     0.9998    1.0000    0.9999    484073
           2     0.5843    0.9885    0.7345      2521
           3     0.9955    0.9508    0.9726     36260
           4     0.8508    0.8194    0.8348     18981
           5     0.9566    0.9955    0.9757      2460
           6     0.9226    0.8516    0.8857     11851
           7     1.0000    0.9965    0.9983    107440
           8     0.9502    0.9595    0.9549     16760
           9     0.9879    0.9854    0.9867     41522
          10     0.8498    0.8604    0.8551     18572

    accuracy                         0.9815    760

Decision Tree

In [7]:
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, classification_report
import time

print("Đang huấn luyện phân lớp Decision Tree (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...")
start_time = time.time()

# Thử nghiệm các giá trị max_depth khác nhau
max_depth_values = [10, 15, 20, 25, 30]

best_depth = None
best_f1 = -1
best_dt_model = None

for depth in max_depth_values:
    # Dùng XGBClassifier với n_estimators=1 để thiết lập giống một Decision Tree chạy bằng GPU
    dt_model = XGBClassifier(
        n_estimators=1,
        learning_rate=1.0,
        max_depth=depth,
        n_jobs=-1,
        random_state=42,
        tree_method='hist',
        device='cuda' # Chạy bằng GPU
    )

    # Huấn luyện mô hình
    dt_model.fit(X_train, y_train)
    
    # Dự đoán trên tập validation và tính Macro F1
    y_valid_pred = dt_model.predict(X_valid)
    current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
    
    print(f"max_depth = {depth:2d} | Validation Macro F1: {current_f1:.4f}")
    
    # Chọn model tốt nhất dựa trên Macro F1 của validation
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_depth = depth
        best_dt_model = dt_model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: max_depth = {best_depth} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print("\n--- Đánh giá mô hình Decision Tree Tốt nhất (max_depth={}) trên tập TEST ---".format(best_depth))
y_test_pred = best_dt_model.predict(X_test)
print(classification_report(y_test, y_test_pred, digits=4))

Đang huấn luyện phân lớp Decision Tree (sử dụng GPU qua XGBoost) và tìm 'max_depth' tốt nhất...
max_depth = 10 | Validation Macro F1: 0.8835
max_depth = 15 | Validation Macro F1: 0.8981
max_depth = 20 | Validation Macro F1: 0.9024
max_depth = 25 | Validation Macro F1: 0.9026
max_depth = 30 | Validation Macro F1: 0.9026

=> Quá trình huấn luyện và tuning hoàn tất trong 34.58 giây.
=> Cấu hình tốt nhất: max_depth = 30 với Validation Macro F1: 0.9026

--- Đánh giá mô hình Decision Tree Tốt nhất (max_depth=30) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.8027    0.8501    0.8257     19851
           1     0.9984    1.0000    0.9992    484073
           2     0.5981    0.9710    0.7402      2521
           3     0.9939    0.9496    0.9712     36260
           4     0.8521    0.8052    0.8280     18981
           5     0.9635    0.9549    0.9592      2460
           6     0.9237    0.8467    0.8835     11851
           7     1.0000    0.9965    

Logistic Regression

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score, classification_report
import time
import numpy as np
import pandas as pd

print("Đang huấn luyện phân lớp Logistic Regression (chạy thực sự 100% trên GPU qua PyTorch) và tìm 'alpha' & 'penalty' tốt nhất...")
start_time = time.time()

# Thử nghiệm các alpha và penalty l1, l2
alpha_values = [1e-3, 1e-4, 1e-5, 1e-6]
penalty_values = ['l1', 'l2']

best_alpha = None
best_penalty = None
best_f1 = -1
best_logreg_model = None

# Khối 1: Chuyển đổi toàn bộ mảng dữ liệu về Numpy (nếu đang ở pandas DataFrame)
X_trn = X_train.values if isinstance(X_train, pd.DataFrame) else X_train
y_trn = y_train.values if isinstance(y_train, pd.Series) else y_train
X_val = X_valid.values if isinstance(X_valid, pd.DataFrame) else X_valid
X_tst = X_test.values if isinstance(X_test, pd.DataFrame) else X_test

# Khối 2: Đưa thẳng lên VRAM của GPU với PyTorch
device = 'cuda'
X_train_t = torch.tensor(X_trn, dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_trn, dtype=torch.long, device=device)
X_valid_t = torch.tensor(X_val, dtype=torch.float32, device=device)
X_test_t = torch.tensor(X_tst, dtype=torch.float32, device=device)

input_dim = X_train_t.shape[1]
num_classes = len(torch.unique(y_train_t))

# Định nghĩa mạng: 1 tầng Linear duy nhất + Hàm phạt Cross Entropy = Logistic Regression
class TorchLogisticRegression(nn.Module):
    def __init__(self, in_features, out_classes):
        super().__init__()
        self.linear = nn.Linear(in_features, out_classes)
        
    def forward(self, x):
        return self.linear(x)

for penalty in penalty_values:
    for alpha in alpha_values:
        model = TorchLogisticRegression(input_dim, num_classes).to(device)
        
        # Nếu l2 thì dùng tham số weight_decay tích hợp sẵn của Optimizer tự tính toán
        weight_decay = alpha if penalty == 'l2' else 0.0
        optimizer = optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)
        criterion = nn.CrossEntropyLoss()
        
        model.train()
        epochs = 100
        batch_size = 8192
        num_samples = X_train_t.shape[0]
        
        for epoch in range(epochs):
            # Xáo trộn index bộ nhớ trực tiếp trên GPU để đảm bảo tốc độ tối đa
            indices = torch.randperm(num_samples, device=device)
            
            for i in range(0, num_samples, batch_size):
                idx = indices[i:i+batch_size]
                batch_X = X_train_t[idx]
                batch_y = y_train_t[idx]
                
                optimizer.zero_grad()
                outputs = model(batch_X)
                loss = criterion(outputs, batch_y)
                
                # Nếu l1 thì cộng tay penalty L1 Norm
                if penalty == 'l1':
                    l1_norm = sum(p.abs().sum() for p in model.parameters())
                    loss = loss + alpha * l1_norm
                    
                loss.backward()
                optimizer.step()
        
        model.eval()
        with torch.no_grad():
            valid_outputs = model(X_valid_t)
            y_valid_pred = torch.argmax(valid_outputs, dim=1).cpu().numpy()
            
        current_f1 = f1_score(y_valid, y_valid_pred, average='macro')
        print(f"penalty = {penalty:2s}, alpha = {alpha:8.6f} | Validation Macro F1: {current_f1:.4f}")
        
        # Chọn model tốt nhất dựa trên Macro F1
        if current_f1 > best_f1:
            best_f1 = current_f1
            best_alpha = alpha
            best_penalty = penalty
            best_logreg_model = model

train_time = time.time() - start_time
print(f"\n=> Quá trình huấn luyện và tuning hoàn tất trong {train_time:.2f} giây.")
print(f"=> Cấu hình tốt nhất: penalty = {best_penalty}, alpha = {best_alpha} với Validation Macro F1: {best_f1:.4f}")

# Đánh giá trên tập test bằng classification report
print(f"\n--- Đánh giá mô hình Logistic Regression Tốt nhất (penalty={best_penalty}, alpha={best_alpha}) trên tập TEST ---")
best_logreg_model.eval()
with torch.no_grad():
    test_outputs = best_logreg_model(X_test_t)
    y_test_pred_final = torch.argmax(test_outputs, dim=1).cpu().numpy()

print(classification_report(y_test, y_test_pred_final, digits=4))

Đang huấn luyện phân lớp Logistic Regression (chạy thực sự 100% trên GPU qua PyTorch) và tìm 'alpha' & 'penalty' tốt nhất...
penalty = l1, alpha = 0.001000 | Validation Macro F1: 0.6792
penalty = l1, alpha = 0.000100 | Validation Macro F1: 0.6825
penalty = l1, alpha = 0.000010 | Validation Macro F1: 0.7978
penalty = l1, alpha = 0.000001 | Validation Macro F1: 0.8302
penalty = l2, alpha = 0.001000 | Validation Macro F1: 0.7587
penalty = l2, alpha = 0.000100 | Validation Macro F1: 0.7457
penalty = l2, alpha = 0.000010 | Validation Macro F1: 0.7354
penalty = l2, alpha = 0.000001 | Validation Macro F1: 0.8204

=> Quá trình huấn luyện và tuning hoàn tất trong 239.34 giây.
=> Cấu hình tốt nhất: penalty = l1, alpha = 1e-06 với Validation Macro F1: 0.8302

--- Đánh giá mô hình Logistic Regression Tốt nhất (penalty=l1, alpha=1e-06) trên tập TEST ---
              precision    recall  f1-score   support

           0     0.5793    0.7998    0.6719     19851
           1     0.9976    1.0000    0